# Week 2 — Initial Implementation

**9 interactive visualizations** (3 per question) using Altair on the French baby names dataset.

| Question | Design | Description |
|---|---|---|
| **1 — Temporal** | 1.2 | Bump chart of ranks by decade |
| | 1.6 | Name × Year heatmap (popularity frieze) |
| | 1.8 | Streamgraph — name families over time |
| **2 — Regional** | 2.1 | Choropleth map by department |
| | 2.3 | Department ranking bar chart |
| | 2.8 | Regional radar / small multiples |
| **3 — Gender** | 3.3 | Diverging bar chart M vs F |
| | 3.6 | Popularity × gender-balance scatterplot |
| | 3.7 | Gender mini-cards (small multiples) |

In [1]:
import pandas as pd
import altair as alt
import numpy as np
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

alt.data_transformers.disable_max_rows()

df = pd.read_csv('../data/dpt2020.csv', sep=';', dtype={'annais': str, 'dpt': str, 'nombre': str})
df = df[df['annais'] != 'XXXX']
df = df[df['preusuel'] != '_PRENOMS_RARES']
df['annais'] = pd.to_numeric(df['annais'], errors='coerce')
df['nombre'] = pd.to_numeric(df['nombre'], errors='coerce')
df = df.dropna(subset=['annais', 'nombre'])
df['annais'] = df['annais'].astype(int)

# Agrégation nationale
nat = df.groupby(['preusuel', 'annais', 'sexe'])['nombre'].sum().reset_index()
nat['decade'] = (nat['annais'] // 10) * 10
nat_dec = nat.groupby(['preusuel', 'decade', 'sexe'])['nombre'].sum().reset_index()
nat_dec['rank'] = nat_dec.groupby(['decade', 'sexe'])['nombre'].rank(ascending=False, method='min')
nat_dec['rank'] = nat_dec['rank'].astype(int)
nat_dec['sex_label'] = nat_dec['sexe'].map({1: 'Male', 2: 'Female'})

top20_names = nat_dec[nat_dec['rank'] <= 20]['preusuel'].unique()
bump_data = nat_dec[nat_dec['preusuel'].isin(top20_names) & (nat_dec['rank'] <= 20)].copy()

# Totaux par prénom et décennie
pop_total = nat.groupby(['preusuel','annais'])['nombre'].sum().reset_index()
top50 = df.groupby('preusuel')['nombre'].sum().nlargest(50).index.tolist()

print(f'{len(df):,} rows — {df["preusuel"].nunique():,} names — {df["annais"].min()}–{df["annais"].max()}')

3,773,985 rows — 16,226 names — 1900–2022


---
# Visualization 1 — Temporal evolution
## 1.2 — Bump chart of ranks

In [2]:
highlight = alt.selection_point(fields=['preusuel'], on='click', toggle=True, empty=True)
sex_input = alt.param(name='sex_param', value='Female',
    bind=alt.binding_radio(options=['Female', 'Male'], name='Sex: '))

# Top 10 seulement pour la lisibilité
bump10 = bump_data[bump_data['rank'] <= 10].copy()
color_scale = alt.Color('preusuel:N', legend=alt.Legend(title='Name (click to highlight)'))
last_dec = int(bump10['decade'].max())

base = alt.Chart(bump10).transform_filter('datum.sex_label === sex_param').encode(
    x=alt.X('decade:O', title='Decade', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('rank:Q', title='Rank', scale=alt.Scale(reverse=True, domain=[1, 10])),
    color=color_scale,
    opacity=alt.condition(highlight, alt.value(1.0), alt.value(0.1)),
    size=alt.condition(highlight, alt.value(4), alt.value(2)),
    detail='preusuel:N',
    tooltip=['preusuel:N', 'decade:O', 'rank:Q', alt.Tooltip('nombre:Q', format=',')]
)

labels = alt.Chart(bump10[bump10['decade'] == last_dec]).transform_filter(
    'datum.sex_label === sex_param'
).mark_text(align='left', dx=7, fontSize=11, fontWeight='bold').encode(
    x=alt.X('decade:O'),
    y=alt.Y('rank:Q', scale=alt.Scale(reverse=True, domain=[1, 10])),
    text='preusuel:N', color=color_scale,
    opacity=alt.condition(highlight, alt.value(1.0), alt.value(0.1))
)

(base.mark_line(point=alt.OverlayMarkDef(size=80), strokeWidth=3) + labels
).add_params(highlight, sex_input).properties(
    width=720, height=400,
    title=alt.Title('1.2 — Top 10 French Baby Names — Rank by Decade',
                    subtitle='Click a line (or legend) to highlight it. Radio to switch M/F.')
)

alt.LayerChart(...)

## 1.6 — Name × Year heatmap (popularity frieze)

In [3]:
# Top 30 noms par nombre total de naissances
top30 = df.groupby('preusuel')['nombre'].sum().nlargest(30).index.tolist()
heat_data = pop_total[pop_total['preusuel'].isin(top30)].copy()
heat_data['decade'] = (heat_data['annais'] // 10) * 10
heat_dec = heat_data.groupby(['preusuel', 'decade'])['nombre'].sum().reset_index()

# Normaliser par nom (max=1) pour comparer les formes
heat_dec['norm'] = heat_dec.groupby('preusuel')['nombre'].transform(lambda x: x / x.max())

# Ordonner les noms par décennie de pic
peak_dec = heat_dec.loc[heat_dec.groupby('preusuel')['nombre'].idxmax(), ['preusuel','decade']]
name_order = peak_dec.sort_values('decade')['preusuel'].tolist()

alt.Chart(heat_dec).mark_rect().encode(
    x=alt.X('decade:O', title='Decade', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('preusuel:N', title='Name', sort=name_order),
    color=alt.Color('norm:Q', scale=alt.Scale(scheme='inferno'), title='Relative popularity'),
    tooltip=['preusuel:N', 'decade:O',
             alt.Tooltip('nombre:Q', title='Births', format=','),
             alt.Tooltip('norm:Q', title='Relative', format='.2f')]
).properties(
    width=700, height=550,
    title=alt.Title('Name × Decade Heatmap — Top 30 French Names',
                    subtitle='Each row = one name. Brighter = more popular. Names sorted by their peak decade.')
)

alt.Chart(...)

## 1.8 — Streamgraph — name families over time

In [4]:
# Top 12 noms féminins + top 12 noms masculins sur la période entière
top_f = nat[nat['sexe']==2].groupby('preusuel')['nombre'].sum().nlargest(12).index.tolist()
top_m = nat[nat['sexe']==1].groupby('preusuel')['nombre'].sum().nlargest(12).index.tolist()

stream_sel = alt.param(name='stream_sex', value='Female',
    bind=alt.binding_radio(options=['Female', 'Male'], name='Sex: '))

stream_f = nat[(nat['sexe']==2) & (nat['preusuel'].isin(top_f))].groupby(['annais','preusuel'])['nombre'].sum().reset_index()
stream_f['sex_label'] = 'Female'
stream_m = nat[(nat['sexe']==1) & (nat['preusuel'].isin(top_m))].groupby(['annais','preusuel'])['nombre'].sum().reset_index()
stream_m['sex_label'] = 'Male'
stream_data = pd.concat([stream_f, stream_m], ignore_index=True)

alt.Chart(stream_data).mark_area().transform_filter(
    'datum.sex_label === stream_sex'
).encode(
    x=alt.X('annais:Q', title='Year', axis=alt.Axis(format='d', labelAngle=0)),
    y=alt.Y('nombre:Q', stack='center', title='Births (centered)', axis=None),
    color=alt.Color('preusuel:N', legend=alt.Legend(title='Name')),
    tooltip=['preusuel:N', alt.Tooltip('annais:Q', title='Year'),
             alt.Tooltip('nombre:Q', title='Births', format=',')]
).add_params(stream_sel).properties(
    width=750, height=420,
    title=alt.Title('Streamgraph — Top 12 Names Over Time',
                    subtitle='Band width = number of births. Use the radio to switch sex.')
)

alt.Chart(...)

---
# Visualization 2 — Regional effects
## 2.1 — Choropleth map by department

In [5]:
# Pré-calcul map_data
total_dpt = df.groupby(['dpt','annais'])['nombre'].sum().reset_index()
total_dpt['decade'] = (total_dpt['annais'] // 10) * 10
total_dpt_dec = total_dpt.groupby(['dpt','decade'])['nombre'].sum().reset_index()
total_dpt_dec.columns = ['dpt','decade','total_births']

name_dpt = df.groupby(['preusuel','dpt','annais'])['nombre'].sum().reset_index()
name_dpt['decade'] = (name_dpt['annais'] // 10) * 10
name_dpt_dec = name_dpt.groupby(['preusuel','dpt','decade'])['nombre'].sum().reset_index()
name_dpt_dec = name_dpt_dec.merge(total_dpt_dec, on=['dpt','decade'], how='left')
name_dpt_dec['rate_per_10k'] = (name_dpt_dec['nombre'] / name_dpt_dec['total_births'] * 10000).round(2)
map_data = name_dpt_dec[name_dpt_dec['preusuel'].isin(top50)].copy()

with open('../data/departements.geojson') as fgeo:
    geo = json.load(fgeo)
geo_inline = alt.InlineData(values=geo['features'], format=alt.DataFormat(type='json'))

top20_map = df.groupby('preusuel')['nombre'].sum().nlargest(20).index.tolist()
decades_map = sorted(map_data['decade'].unique().tolist())

def make_choropleth(name, decade):
    filtered = map_data[(map_data['preusuel']==name) & (map_data['decade']==decade)][['dpt','rate_per_10k','nombre']]
    return alt.Chart(geo_inline).mark_geoshape(stroke='white', strokeWidth=0.5
    ).transform_lookup(
        lookup='properties.code',
        from_=alt.LookupData(alt.InlineData(values=filtered.to_dict(orient='records')), 'dpt', ['rate_per_10k','nombre'])
    ).encode(
        color=alt.Color('rate_per_10k:Q', scale=alt.Scale(scheme='blues', domainMin=0), title='Births / 10k'),
        tooltip=[alt.Tooltip('properties.nom:N', title='Department'),
                 alt.Tooltip('rate_per_10k:Q', title='Rate / 10k', format='.1f'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')]
    ).project('mercator').properties(width=620, height=520, title=f'{name}  —  {decade}s')

name_dd = widgets.Dropdown(options=sorted(top20_map), value='MARIE', description='Name:')
decade_dd = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
out_map = widgets.Output()
def on_map_change(_):
    with out_map:
        clear_output(wait=True)
        display(make_choropleth(name_dd.value, decade_dd.value))
name_dd.observe(on_map_change, names='value')
decade_dd.observe(on_map_change, names='value')
display(widgets.HBox([name_dd, decade_dd]), out_map)
on_map_change(None)

Output()

## 2.3 — Department ranking bar chart

In [6]:
def make_ranking(name, decade, n=20):
    filtered = map_data[(map_data['preusuel']==name) & (map_data['decade']==decade)].copy()
    filtered = filtered.dropna(subset=['rate_per_10k'])

    # Noms complets incluant DOM-TOM
    dpt_names_full = {f['properties']['code']: f['properties']['nom'] for f in geo['features']}
    dpt_names_full.update({
        '971': 'Guadeloupe', '972': 'Martinique', '973': 'Guyane',
        '974': 'La Réunion', '976': 'Mayotte', '20': 'Corse (ancien)'
    })

    filtered['dept_name'] = filtered['dpt'].map(dpt_names_full).fillna('Dept ' + filtered['dpt'])
    filtered = filtered.sort_values('rate_per_10k', ascending=False).head(n)

    return alt.Chart(filtered).mark_bar(color='#2196F3').encode(
        x=alt.X('rate_per_10k:Q', title='Births per 10 000'),
        y=alt.Y('dept_name:N', sort='-x', title='Department'),
        tooltip=[alt.Tooltip('dept_name:N', title='Department'),
                 alt.Tooltip('rate_per_10k:Q', title='Rate / 10k', format='.1f'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')]
    ).properties(width=600, height=480,
                  title=f'2.3 — Top {n} departments for {name} — {decade}s')

name_r = widgets.Dropdown(options=sorted(top20_map), value='MARIE', description='Name:')
decade_r = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
out_r = widgets.Output()
def on_r_change(_):
    with out_r:
        clear_output(wait=True)
        display(make_ranking(name_r.value, decade_r.value))
name_r.observe(on_r_change, names='value')
decade_r.observe(on_r_change, names='value')
display(widgets.HBox([name_r, decade_r]), out_r)
on_r_change(None)

Output()

## 2.8 — Regional flower / small multiples (top regions)

In [7]:
# Mapping département → région (simplifié)
region_map = {
    'Île-de-France':   ['75','77','78','91','92','93','94','95'],
    'Bretagne':        ['22','29','35','56'],
    'PACA':            ['04','05','06','13','83','84'],
    'Auvergne-RA':     ['01','03','07','15','26','38','42','43','63','69','73','74'],
    'Occitanie':       ['09','11','12','30','31','32','34','46','48','65','66','81','82'],
    'Nouvelle-Aquit.': ['16','17','19','23','24','33','40','47','64','79','86','87'],
    'Grand-Est':       ['08','10','51','52','54','55','57','67','68','88'],
    'Normandie':       ['14','27','50','61','76'],
    'Hauts-de-France': ['02','59','60','62','80'],
    'Pays-de-la-Loire':['44','49','53','72','85'],
}
dpt_to_reg = {dpt: reg for reg, dpts in region_map.items() for dpt in dpts}

map_data['region'] = map_data['dpt'].map(dpt_to_reg)
reg_data = map_data.dropna(subset=['region'])
reg_agg = reg_data.groupby(['preusuel','region','decade']).agg(
    nombre=('nombre','sum'), total_births=('total_births','sum')
).reset_index()
reg_agg['rate_per_10k'] = (reg_agg['nombre'] / reg_agg['total_births'] * 10000).round(2)

def make_flower(name, decade):
    d = reg_agg[(reg_agg['preusuel']==name) & (reg_agg['decade']==decade)].copy()
    regions = sorted(region_map.keys())
    return alt.Chart(d).mark_bar(color='#9C27B0').encode(
        x=alt.X('region:N', sort=regions, title=None,
                 axis=alt.Axis(labelAngle=-35, labelFontSize=10)),
        y=alt.Y('rate_per_10k:Q', title='Births per 10 000'),
        tooltip=[alt.Tooltip('region:N', title='Region'),
                 alt.Tooltip('rate_per_10k:Q', title='Rate / 10k', format='.1f'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')]
    ).properties(width=650, height=380, title=f'Regional popularity of {name} — {decade}s')

name_fl = widgets.Dropdown(options=sorted(top20_map), value='MARIE', description='Name:')
decade_fl = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
out_fl = widgets.Output()
def on_fl_change(_):
    with out_fl:
        clear_output(wait=True)
        display(make_flower(name_fl.value, decade_fl.value))
name_fl.observe(on_fl_change, names='value')
decade_fl.observe(on_fl_change, names='value')
display(widgets.HBox([name_fl, decade_fl]), out_fl)
on_fl_change(None)

Output()

---
# Visualization 3 — Gender effects
## 3.3 — Diverging bar chart M vs F

In [8]:
# Prénoms mixtes : sélectionner des noms connus avec un total significatif
gender_nat = nat.groupby(['preusuel','sexe'])['nombre'].sum().reset_index()
g_pivot = gender_nat.pivot_table(index='preusuel', columns='sexe', values='nombre', fill_value=0).reset_index()
g_pivot.columns.name = None
g_pivot = g_pivot.rename(columns={1:'male', 2:'female'})
if 'male' not in g_pivot.columns: g_pivot['male'] = 0
if 'female' not in g_pivot.columns: g_pivot['female'] = 0
g_pivot['total'] = g_pivot['male'] + g_pivot['female']
g_pivot['female_ratio'] = g_pivot['female'] / g_pivot['total']
g_pivot['mixedness'] = (g_pivot['female_ratio'] - 0.5).abs()

# Garder uniquement les prénoms avec au moins 5000 naissances totales (noms reconnus)
mixed = g_pivot[g_pivot['total'] >= 5000].copy()
top_mixed = mixed.nsmallest(25, 'mixedness')['preusuel'].tolist()
div_data = g_pivot[g_pivot['preusuel'].isin(top_mixed)].sort_values('female_ratio').copy()

# Normaliser : afficher le % plutôt que le nombre brut
div_data['male_pct'] = -(div_data['male'] / div_data['total'] * 100).round(1)
div_data['female_pct'] = (div_data['female'] / div_data['total'] * 100).round(1)

div_long = pd.melt(div_data, id_vars='preusuel',
                    value_vars=['male_pct','female_pct'],
                    var_name='sex', value_name='pct')
div_long['sex_label'] = div_long['sex'].map({'male_pct':'Male','female_pct':'Female'})
div_long['pct_abs'] = div_long['pct'].abs()

alt.Chart(div_long).mark_bar().encode(
    x=alt.X('pct:Q', title='← % Male    |    % Female →',
             axis=alt.Axis(labelExpr='abs(datum.value) + "%"'), scale=alt.Scale(domain=[-100, 100])),
    y=alt.Y('preusuel:N', sort=div_data['preusuel'].tolist(), title='Name'),
    color=alt.Color('sex_label:N',
                    scale=alt.Scale(domain=['Male','Female'], range=['#4878cf','#d65f5f']),
                    title='Sex'),
    tooltip=['preusuel:N', 'sex_label:N',
             alt.Tooltip('pct_abs:Q', title='%', format='.1f')]
).properties(width=650, height=540,
              title=alt.Title('3.3 — 25 Most Gender-Mixed Names (% of births)',
                              subtitle='Names with ≥5000 births total. Sorted by female ratio.'))

alt.Chart(...)

## 3.6 — Popularity × gender-balance scatterplot

In [9]:
gender_dec = nat.groupby(['preusuel','sexe','decade'])['nombre'].sum().reset_index()
g_piv2 = gender_dec.pivot_table(index=['preusuel','decade'], columns='sexe', values='nombre', fill_value=0).reset_index()
g_piv2.columns.name = None
g_piv2 = g_piv2.rename(columns={1:'male', 2:'female'})
if 'male' not in g_piv2.columns: g_piv2['male'] = 0
if 'female' not in g_piv2.columns: g_piv2['female'] = 0
g_piv2['total'] = g_piv2['male'] + g_piv2['female']
g_piv2['female_ratio'] = g_piv2['female'] / g_piv2['total']
g_piv2['gender_type'] = 'Mixed (both)'
g_piv2.loc[g_piv2['female_ratio'] > 0.8, 'gender_type'] = 'Mostly female'
g_piv2.loc[g_piv2['female_ratio'] < 0.2, 'gender_type'] = 'Mostly male'
g_embed = g_piv2[g_piv2['total'] >= 500].copy()
g_embed['decade'] = g_embed['decade'].astype(int)

decade_slider = alt.param(name='decade_scatter', value=2010,
    bind=alt.binding_range(min=int(g_embed['decade'].min()), max=int(g_embed['decade'].max()),
                           step=10, name='Decade: '))

scatter = alt.Chart(g_embed).mark_circle(opacity=0.65).transform_filter(
    'datum.decade === decade_scatter'
).encode(
    x=alt.X('total:Q', scale=alt.Scale(type='log'), title='Total births (log scale)'),
    y=alt.Y('female_ratio:Q', scale=alt.Scale(domain=[0,1]),
             title='Female ratio  (0 = all male · 1 = all female)'),
    size=alt.Size('total:Q', scale=alt.Scale(range=[30,600]), legend=None),
    color=alt.Color('gender_type:N',
        scale=alt.Scale(domain=['Mostly male','Mixed (both)','Mostly female'],
                        range=['#4878cf','#6acc65','#d65f5f']), title='Gender type'),
    tooltip=['preusuel:N', alt.Tooltip('decade:Q', title='Decade'),
             alt.Tooltip('total:Q', title='Total births', format=','),
             alt.Tooltip('female_ratio:Q', title='Female ratio', format='.2f')]
).add_params(decade_slider).properties(
    width=720, height=460,
    title=alt.Title('Name Popularity vs. Gender Balance — by Decade',
                    subtitle='Each dot = one name. Drag the slider to change decade.')
)

hline = alt.Chart({'values': [{'y': 0.5}]}).mark_rule(
    strokeDash=[5,3], color='gray', opacity=0.6).encode(y='y:Q')

(scatter + hline)

alt.LayerChart(...)

## 3.7 — Gender mini-cards (small multiples)

In [10]:
# Sélection diverse : top 6 M + top 6 F des années 2000-2020 + 12 prénoms mixtes connus
top6m = nat[(nat['sexe']==1) & (nat['annais']>=2000)].groupby('preusuel')['nombre'].sum().nlargest(6).index.tolist()
top6f = nat[(nat['sexe']==2) & (nat['annais']>=2000)].groupby('preusuel')['nombre'].sum().nlargest(6).index.tolist()
mixed_known = ['CAMILLE','DOMINIQUE','CLAUDE','ALEXIS','CHARLIE','ANDREA',
               'SACHA','EDEN','LOUISON','CHARLIE','YAEL','SASHA']
mixed_known = [n for n in mixed_known if n not in top6m + top6f][:12]
selected24 = top6m + top6f + mixed_known

cards_data = g_piv2[g_piv2['preusuel'].isin(selected24)].copy()
cards_long = pd.melt(cards_data, id_vars=['preusuel','decade'],
                      value_vars=['male','female'], var_name='sex', value_name='births')
cards_long['sex_label'] = cards_long['sex'].map({'male':'Male','female':'Female'})
totals = cards_long.groupby(['preusuel','decade'])['births'].sum().rename('total').reset_index()
cards_long = cards_long.merge(totals, on=['preusuel','decade'])
cards_long['pct'] = (cards_long['births'] / cards_long['total'] * 100).round(1)
cards_2010 = cards_long[cards_long['decade'] == 2010].copy()
cards_2010 = cards_2010[cards_2010['preusuel'].isin(selected24)]

alt.Chart(cards_2010).mark_bar().encode(
    x=alt.X('sex_label:N', title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('pct:Q', title='%', scale=alt.Scale(domain=[0,100])),
    color=alt.Color('sex_label:N',
        scale=alt.Scale(domain=['Male','Female'], range=['#4878cf','#d65f5f']),
        legend=None),
    tooltip=['preusuel:N', 'sex_label:N',
             alt.Tooltip('births:Q', format=','),
             alt.Tooltip('pct:Q', title='%', format='.1f')]
).facet(
    facet=alt.Facet('preusuel:N', title=None), columns=6
).properties(
    title=alt.Title('3.7 — Gender Mini-Cards (decade 2010)',
                    subtitle='Top 6 male + top 6 female (2000–2020) + 12 mixed names. Each card = % M vs F.')
)

alt.FacetChart(...)